# Full manual lineage: Drug shortage listings

Run each stage and inspect its output before continuing. The pinned 2026-09-26 publisher response and verified rebuild are supplied. Set `USE_PINNED_RAW = False` only for a deliberate fresh-source run; its values may differ. All processing and formulas are visible in this notebook. No portfolio module, launcher, or configuration file is required.


## 1 Source

Inspect the publisher URL and query below: https://api.fda.gov/drug/shortages.json. The included raw response is a dated publisher payload with a SHA-256 digest, not the unsupported older portfolio snapshot. To acquire a new response, set `USE_PINNED_RAW = False` before running.

**Check before continuing:** publisher, selection, project folder, and pinned-response location must match this project; any new response must be preserved with its retrieval metadata.


In [1]:
import hashlib
import json
from datetime import UTC, date, datetime
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
import requests
from IPython.display import display

SLUG = "05_drug_safety"
FOLDER = SLUG.split("_", 1)[1]
SOURCE_URL = "https://api.fda.gov/drug/shortages.json"
SNAPSHOT_DATE_OVERRIDE = None  # Leave None for the dated publisher response.
USE_PINNED_RAW = True
ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
             if (candidate / "projects" / FOLDER / "README.md").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the extracted portfolio.")
PROJECT = ROOT / "projects" / FOLDER
RAW_DIR = PROJECT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
FETCH_LOG = []
PINNED_RESPONSES = {'cd019c9dd823611e': '8e0cda58e3b53046e11f73919c060911cca841ed4d848eb684d7709a3f33fd99', 'cf9b74ee336d9b84': '3b35adcfba6cda8d40587a9d3b9bda1995aa183233b0dd29d930bd3f73ae1264'}
print("Project:", PROJECT.relative_to(ROOT).as_posix())
print("Publisher:", SOURCE_URL)
print("Pinned response folder:", RAW_DIR.relative_to(ROOT).as_posix())

"""Cross-sectional FDA listing surveillance, not time-to-resolution inference."""


import hashlib
import json

import numpy as np
import pandas as pd


def normalize_publisher_records(records: list[dict]) -> pd.DataFrame:
    """Retain distinct publisher presentations, removing only exact JSON repeats.

    The content digest identifies a record within this snapshot, not a stable
    episode across refreshes. Package/company alone is not a unique source key.
    Raw responses retain any repeated records for reconciliation.
    """
    rows = []
    seen = set()
    fields = (
        "package_ndc",
        "generic_name",
        "company_name",
        "status",
        "availability",
        "dosage_form",
        "presentation",
        "update_type",
        "initial_posting_date",
        "update_date",
    )
    for record in records:
        canonical = json.dumps(record, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
        record_id = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
        if record_id in seen:
            continue
        seen.add(record_id)
        row = {field: record.get(field) for field in fields}
        category = record.get("therapeutic_category") or []
        row["therapeutic_category"] = (
            ", ".join(category) if isinstance(category, list) else category
        )
        row["record_id"] = record_id
        rows.append(row)
    result = pd.DataFrame(rows)
    result.attrs["publisher_rows"] = len(records)
    result.attrs["exact_duplicate_rows_removed"] = len(records) - len(rows)
    return result


def listing_metrics(frame: pd.DataFrame, *, as_of: str | pd.Timestamp) -> pd.DataFrame:
    """Age starts at public posting, not an assumed onset of national shortage.

    Current listings remain unresolved at observation. Resolved/discontinued
    entries have no inferred event time. A prevalent cross-section cannot support
    a recovery survival curve: historical resolved episodes may be absent and
    long-lived entries are preferentially observed.
    """
    cutoff = pd.Timestamp(as_of)
    if pd.isna(cutoff):
        raise ValueError("An explicit valid snapshot date is required")
    if cutoff.tzinfo is not None:
        cutoff = cutoff.tz_convert("UTC").tz_localize(None)
    cutoff = cutoff.normalize()
    x = frame.copy(deep=True)
    for column in ["company_name", "generic_name", "package_ndc"]:
        x[column] = x[column].astype("string").str.strip().replace("", pd.NA)
    if "record_id" in x:
        if x["record_id"].isna().any() or x["record_id"].duplicated().any():
            raise ValueError("Duplicate or missing publisher record identifiers")
    elif x.duplicated(["package_ndc", "company_name"]).any():
        raise ValueError("Duplicate package/company records require explicit reconciliation")
    else:
        # Legacy snapshots and fixtures lack publisher JSON; label their local
        # row digest separately through acquisition provenance, never as FDA IDs.
        x["record_id"] = [
            hashlib.sha256(json.dumps(row, sort_keys=True, default=str).encode()).hexdigest()
            for row in x.to_dict("records")
        ]
    status = x["status"].astype("string").str.strip().str.casefold()
    x["active_flag"] = status.eq("current").fillna(False).astype(int)
    x["status_known_flag"] = status.isin(
        ["current", "resolved", "to be discontinued", "discontinued"]
    ).astype(int)
    for column in ["initial_posting_date", "update_date"]:
        x[column] = pd.to_datetime(x[column], errors="coerce", format="mixed")
    x["snapshot_date"] = cutoff
    valid_posting = x["initial_posting_date"].notna() & x["initial_posting_date"].le(cutoff)
    x["posting_date_valid_flag"] = valid_posting.astype(int)
    x["listing_age_days"] = (cutoff - x["initial_posting_date"]).dt.days.where(
        valid_posting & x["active_flag"].eq(1)
    )
    x["unresolved_at_snapshot_flag"] = x["active_flag"]
    x["resolution_time_days"] = np.nan
    x["manufacturer_missing_flag"] = x["company_name"].isna().astype(int)
    active = x.loc[x["active_flag"].eq(1)]
    counts = x["generic_name"].map(active.groupby("generic_name")["company_name"].nunique())
    x["listed_manufacturer_count"] = counts.where(x["generic_name"].notna())
    # One known company plus missing companies is not evidence of one source.
    missing = x["generic_name"].map(
        active.groupby("generic_name")["manufacturer_missing_flag"].sum()
    )
    x["one_listed_manufacturer_flag"] = (
        counts.eq(1).astype("Int64").where(missing.eq(0) & x["generic_name"].notna())
    )
    return x


def concentration_sensitivity(frame: pd.DataFrame) -> pd.DataFrame:
    """Bounds for package-record HHI, not production capacity or market shares.

    Lower bound assigns each unknown company record to a distinct new company;
    upper bound assigns all unknown records to the largest known company. The
    complete-case HHI is reported separately because its denominator differs.
    Multiple packages per company can inflate record exposure, so this statistic
    must never be called manufacturer market concentration.
    """
    rows = []
    active = frame.loc[frame["active_flag"].eq(1)]
    for generic, group in active.groupby("generic_name", dropna=False):
        counts = group["company_name"].dropna().value_counts().to_numpy(dtype=float)
        unknown = int(group["company_name"].isna().sum())
        total = len(group)
        sum_squares = float(np.square(counts).sum())
        largest = float(counts.max()) if len(counts) else 0.0
        known = total - unknown
        rows.append(
            {
                "generic_name": generic,
                "active_package_records": total,
                "known_manufacturers": len(counts),
                "unknown_company_records": unknown,
                "known_record_hhi": sum_squares / known**2 if known else np.nan,
                "record_hhi_lower": (sum_squares + unknown) / total**2,
                "record_hhi_upper": (sum_squares - largest**2 + (largest + unknown) ** 2)
                / total**2,
            }
        )
    return pd.DataFrame(rows)

def preserve_response(
    response: requests.Response,
    raw_root: Path,
    *,
    filename: str,
    publisher: str,
    **metadata: object,
) -> Path:
    response.raise_for_status()
    if Path(filename).name != filename:
        raise ValueError("Snapshot filename must be a basename")
    payload = response.content
    if not payload:
        raise ValueError("Cannot preserve an empty source snapshot")
    digest = hashlib.sha256(payload).hexdigest()
    directory = raw_root / digest
    directory.mkdir(parents=True, exist_ok=True)
    destination = directory / filename
    if destination.exists() and hashlib.sha256(destination.read_bytes()).hexdigest() != digest:
        raise ValueError("Content-addressed snapshot collision")
    if not destination.exists():
        destination.write_bytes(payload)
    record = {
        **metadata,
        "publisher": publisher,
        "direct_url": response.url,
        "sha256": digest,
        "bytes": len(payload),
        "accessed_at": datetime.now(UTC).isoformat(),
    }
    # Response headers are deliberately excluded from public provenance.
    metadata_path = directory / "metadata.json"
    if metadata_path.exists():
        saved = json.loads(metadata_path.read_text(encoding="utf-8"))
        if saved.get("sha256") != digest or saved.get("bytes") != len(payload):
            raise ValueError("Content-addressed snapshot metadata mismatch")
    else:
        metadata_path.write_text(json.dumps(record, indent=2), encoding="utf-8")
    return destination

def project_dir(slug):
    return ROOT / "projects" / slug.split("_", 1)[1]

def _get(url, **kwargs):
    prepared = requests.Request("GET", url, params=kwargs.get("params")).prepare()
    request_key = hashlib.sha256(prepared.url.encode("utf-8")).hexdigest()[:16]
    folder = RAW_DIR / "publisher_responses" / request_key
    if USE_PINNED_RAW:
        digest = PINNED_RESPONSES.get(request_key)
        if digest is None:
            raise FileNotFoundError("This publisher request has no pinned response")
        if len(digest) != 64 or any(ch not in "0123456789abcdef" for ch in digest):
            raise ValueError("Invalid pinned source checksum")
        body = (folder / f"{digest}.body").read_bytes()
        if hashlib.sha256(body).hexdigest() != digest:
            raise ValueError("Pinned publisher response checksum mismatch")
        metadata = json.loads((folder / f"{digest}.json").read_text(encoding="utf-8"))
        response = requests.Response()
        response.status_code = 200
        response.url = prepared.url
        response._content = body
    else:
        response = requests.get(url, timeout=90, **kwargs)
        response.raise_for_status()
        body = response.content
        if not body:
            raise ValueError("Publisher returned an empty response")
        digest = hashlib.sha256(body).hexdigest()
        folder.mkdir(parents=True, exist_ok=True)
        destination = folder / f"{digest}.body"
        if not destination.exists():
            destination.write_bytes(body)
        metadata = {"url": response.url, "sha256": digest, "bytes": len(body),
                    "accessed_at": datetime.now(UTC).isoformat()}
        (folder / f"{digest}.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    FETCH_LOG.append(metadata)
    return response

def _openfda_shortages() -> pd.DataFrame:

    results = []
    total = None
    updated = None
    while total is None or len(results) < total:
        response = _get(
            "https://api.fda.gov/drug/shortages.json", params={"limit": 1000, "skip": len(results)}
        )
        payload = response.json()
        observed_total = payload["meta"]["results"]["total"]
        observed_updated = payload["meta"].get("last_updated")
        if total is not None and (observed_total != total or observed_updated != updated):
            raise ValueError(
                "FDA publication changed during pagination; discard this incomplete acquisition"
            )
        total, updated = observed_total, observed_updated
        page = payload["results"]
        if not page:
            raise ValueError("FDA returned an empty page before count reconciliation")
        preserve_response(
            response,
            project_dir("05_drug_safety") / "data" / "raw",
            filename=f"shortages_{len(results):05d}.json",
            publisher="US Food and Drug Administration",
            source_url="https://open.fda.gov/apis/drug/drugshortages/",
            coverage=f"Listing snapshot; publisher update {updated}",
            grain="Publisher shortage presentation record",
            refresh_frequency="Publisher updates",
            reuse_basis="Public openFDA data; retain FDA disclaimer",
            attribution="US FDA openFDA Drug Shortages",
            limitations="Not medical advice; update dates are not resolution dates",
            publisher_total=total,
        )
        results.extend(page)
        if len(results) > total or (len(results) < total and len(results) > 25000):
            raise ValueError("FDA count reconciliation or pagination limit failed")

    return normalize_publisher_records(results)


Project: projects/drug_safety
Publisher: https://api.fda.gov/drug/shortages.json
Pinned response folder: projects/drug_safety/data/raw


## 2 Import

Run the import cell. It verifies the pinned response hash (or retrieves a new publisher response), parses it, and displays row count, columns, and a preview.

**Check before continuing:** no parse or source-integrity error; the imported table is nonempty, has the expected fields and grain, and the displayed source date matches the saved response metadata.


In [2]:
raw = _openfda_shortages()
if raw.empty:
    raise ValueError("Publisher returned no source records")
SNAPSHOT_DATE = SNAPSHOT_DATE_OVERRIDE or (
    str(FETCH_LOG[0].get("accessed_at") or "")[:10] if FETCH_LOG else date.today().isoformat()
)
raw.to_csv(RAW_DIR / "source_snapshot.csv", index=False)
print("Snapshot date:", SNAPSHOT_DATE)
print("Publisher responses:", len(FETCH_LOG))
print("Imported rows:", len(raw), "columns:", len(raw.columns))
display(raw.head(3))


Snapshot date: 2026-09-26
Publisher responses: 2
Imported rows: 1600 columns: 12


,package_ndc,generic_name,company_name,status,availability,dosage_form,presentation,update_type,initial_posting_date,update_date,therapeutic_category,record_id
0,0480-3667-22,Liraglutide Injection,"Novo Nordisk, Inc.",Current,Available,Injection,"Liraglutide, Injection, 6 mg/1 mL (NDC 0480-36...",Reverified,07/18/2023,09/22/2026,Endocrinology/Metabolism,4f401b0a6c2fdc8f11dd5a7109275209e1f7bbc149b8a3...
1,59651-722-99,Clonazepam Tablet,Aurobindo Pharma USA,Current,Available,Tablet,"Clonazepam, Tablet, .5 mg (NDC 59651-722-99)",Reverified,03/10/2023,06/02/2026,"Neurology, Pediatric, Psychiatry",a778609e5338736095337cff8701138a87ee6ddad693e0...
2,0409-3365-10,Hydromorphone Hydrochloride Injection,"Hospira, Inc., a Pfizer Company",Current,Available,Injection,"Hydromorphone Hydrochloride, Injection, 2 mg/1...",Reverified,10/31/2017,09/22/2026,Analgesia/Addiction,a05667d0de46a60398ad372c9b0d6db8cace054a37e581...


## 3 Transform and clean

Run the original Drug shortage listings transformation, embedded below. Inspect the fields, grain, and missing values before continuing.

**Check before continuing: inspect the cleaned preview, required fields, record grain, types, and missing values. Stop if a key or type rule fails.**


In [3]:
from dataclasses import asdict, dataclass

import pandas as pd


@dataclass(frozen=True)
class QualityReport:
    rows: int
    columns: int
    duplicate_rows: int
    missing_cells: int
    missing_rate: float
    passed: bool
    notes: list[str]


class DataQualityError(ValueError):
    """Raised when a dataset fails the required release checks."""


def profile(df: pd.DataFrame, key: str | None = None) -> dict:
    notes: list[str] = []
    duplicate_rows = int(df.duplicated().sum())
    missing_cells = int(df.isna().sum().sum())
    total_cells = max(df.shape[0] * df.shape[1], 1)
    if df.empty:
        notes.append("Dataset is empty")
    if key:
        if key not in df:
            notes.append(f"Required key '{key}' is missing")
        else:
            if df[key].isna().any():
                notes.append(f"Key '{key}' contains missing values")
            if df[key].duplicated().any():
                notes.append(f"Key '{key}' is not unique")
    if duplicate_rows:
        notes.append(f"Found {duplicate_rows} duplicate rows")
    report = QualityReport(
        rows=len(df),
        columns=len(df.columns),
        duplicate_rows=duplicate_rows,
        missing_cells=missing_cells,
        missing_rate=round(missing_cells / total_cells, 6),
        passed=not notes,
        notes=notes,
    )
    return asdict(report)


def require_passed(report: dict, project: str) -> None:
    if not report["passed"]:
        details = "; ".join(report["notes"]) or "unspecified quality failure"
        raise DataQualityError(f"Quality checks failed for {project}: {details}")


def require_columns(df: pd.DataFrame, columns: list[str]) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise ValueError(f"Required columns missing: {', '.join(missing)}")

def _snake(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (
        out.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return out

def drug_shortages(df: pd.DataFrame, *, as_of: str | pd.Timestamp) -> pd.DataFrame:
    x = _snake(df)
    require_columns(
        x,
        [
            "package_ndc",
            "generic_name",
            "company_name",
            "status",
            "initial_posting_date",
            "update_date",
        ],
    )
    x = listing_metrics(x, as_of=as_of)
    x["availability_group"] = np.select(
        [
            x["availability"].astype(str).str.contains("unavailable", case=False, na=False),
            x["availability"].astype(str).str.contains("available", case=False, na=False),
        ],
        ["Unavailable", "Some availability"],
        default="Unclear",
    )
    # Legacy extract column names retained for migration compatibility only.
    # These measure active LISTING age and observed listed-company redundancy.
    x["shortage_age_days"] = x["listing_age_days"]
    x["manufacturer_count"] = x["listed_manufacturer_count"]
    x["single_source_flag"] = x["one_listed_manufacturer_flag"]
    x["category_active_count"] = x.groupby("therapeutic_category")["active_flag"].transform("sum")
    x["concentration_pressure"] = (
        1 / x["manufacturer_count"].where(x["manufacturer_count"] > 0)
    ).round(4)
    unavailable = x["availability_group"].eq("Unavailable").astype(int)
    x["escalation_score"] = (
        (
            100
            * (
                0.35 * x["active_flag"]
                + 0.25 * unavailable
                + 0.25 * x["concentration_pressure"]
                + 0.15 * x["shortage_age_days"].clip(0, 730).div(730)
            )
        )
        .where(x["active_flag"].eq(1))
        .round(1)
    )
    x["escalation_band"] = pd.cut(
        x["escalation_score"], [-np.inf, 35, 60, np.inf], labels=["Monitor", "Review", "Escalate"]
    )
    keep = [
        "record_id",
        "snapshot_date",
        "package_ndc",
        "generic_name",
        "company_name",
        "status",
        "availability",
        "availability_group",
        "therapeutic_category",
        "dosage_form",
        "presentation",
        "update_type",
        "initial_posting_date",
        "update_date",
        "shortage_age_days",
        "active_flag",
        "manufacturer_count",
        "single_source_flag",
        "category_active_count",
        "concentration_pressure",
        "escalation_score",
        "escalation_band",
    ]
    return x.reindex(columns=keep)

clean = drug_shortages(raw, as_of=SNAPSHOT_DATE)
print("Clean rows:", len(clean), "columns:", len(clean.columns))
display(clean.head(3))


Clean rows: 1600 columns: 22


,record_id,snapshot_date,package_ndc,generic_name,company_name,status,availability,availability_group,therapeutic_category,dosage_form,...,initial_posting_date,update_date,shortage_age_days,active_flag,manufacturer_count,single_source_flag,category_active_count,concentration_pressure,escalation_score,escalation_band
0,4f401b0a6c2fdc8f11dd5a7109275209e1f7bbc149b8a3...,2026-09-26,0480-3667-22,Liraglutide Injection,"Novo Nordisk, Inc.",Current,Available,Some availability,Endocrinology/Metabolism,Injection,...,2023-07-18,2026-09-22,1166.0,1,3.0,0,9,0.3333,58.3,Review
1,a778609e5338736095337cff8701138a87ee6ddad693e0...,2026-09-26,59651-722-99,Clonazepam Tablet,Aurobindo Pharma USA,Current,Available,Some availability,"Neurology, Pediatric, Psychiatry",Tablet,...,2023-03-10,2026-06-02,1296.0,1,6.0,0,40,0.1667,54.2,Review
2,a05667d0de46a60398ad372c9b0d6db8cace054a37e581...,2026-09-26,0409-3365-10,Hydromorphone Hydrochloride Injection,"Hospira, Inc., a Pfizer Company",Current,Available,Some availability,Analgesia/Addiction,Injection,...,2017-10-31,2026-09-22,3252.0,1,4.0,0,91,0.2500,56.2,Review


## 4 Calculate

Run the original analytical formulas, embedded below. Inspect the concentration_analysis output before continuing.

**Check before continuing: confirm the named result table is nonempty and its labels and units match the project description. Do not treat a fresh snapshot as the historical baseline.**


In [4]:
"""Decision-specific analytical routines used by notebooks and release checks."""


import re
from dataclasses import dataclass

import numpy as np
import pandas as pd

SEED = 20260904


@dataclass(frozen=True)
class AnalysisSpec:
    group: str
    metric: str
    score: str
    entity: str
    date: str | None
    higher_is_risk: bool = True


SPECS = {
    "01_consumer_complaints": AnalysisSpec(
        "product", "timely_flag", "response_risk_score", "complaint_id", "date_received", False
    ),
    "03_sec_financial_health": AnalysisSpec(
        "bank_name", "noncurrent_loan_pct", "composite_stress_score", "cert", "report_date"
    ),
    "04_hospital_readmissions": AnalysisSpec(
        "measure_name", "shrunk_readmission_ratio", "priority_score", "facility_id", None
    ),
    "05_drug_safety": AnalysisSpec(
        "therapeutic_category",
        "shortage_age_days",
        "escalation_score",
        "package_ndc",
        "update_date",
    ),
    "07_electricity_affordability": AnalysisSpec(
        "state", "sales_yoy_pct", "load_pressure_index", "state", "period"
    ),
    "08_power_plant_emissions": AnalysisSpec(
        "fuel", "co2_lb_mwh", "transition_priority_score", "plant_id", None
    ),
    "10_semiconductor_supply_chain": AnalysisSpec(
        "metric", "yoy_pct", "pressure_zscore", "series_id", "period"
    ),
}

def data_audit(frame: pd.DataFrame, entity: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "check": [
                "rows",
                "columns",
                "duplicate_entity_rows",
                "missing_cells",
                "numeric_columns",
            ],
            "value": [
                len(frame),
                frame.shape[1],
                int(frame.duplicated(entity).sum()) if entity in frame else 0,
                int(frame.isna().sum().sum()),
                len(frame.select_dtypes(include="number").columns),
            ],
        }
    )

def segment_scorecard(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    grouped = (
        frame.groupby(spec.group, dropna=False)
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            metric_median=(spec.metric, "median"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
    )
    grouped["metric_gap_vs_total"] = grouped["metric_mean"] - frame[spec.metric].mean()
    grouped["risk_rank"] = (
        grouped["risk_score"].rank(method="dense", ascending=False).astype("Int64")
    )
    return grouped.sort_values(["risk_rank", "records"], ascending=[True, False])

def bootstrap_top_gap(frame: pd.DataFrame, spec: AnalysisSpec, draws: int = 1500) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    scorecard = segment_scorecard(frame, spec)
    focus = scorecard.iloc[0][spec.group]
    a = frame.loc[frame[spec.group] == focus, spec.metric].dropna().to_numpy(float)
    b = frame.loc[frame[spec.group] != focus, spec.metric].dropna().to_numpy(float)
    if min(len(a), len(b)) < 2:
        return pd.DataFrame(
            {"focus_group": [focus], "gap": [np.nan], "ci_low": [np.nan], "ci_high": [np.nan]}
        )
    gaps = np.empty(draws)
    for i in range(draws):
        gaps[i] = (
            rng.choice(a, len(a), replace=True).mean() - rng.choice(b, len(b), replace=True).mean()
        )
    return pd.DataFrame(
        {
            "focus_group": [focus],
            "gap": [a.mean() - b.mean()],
            "ci_low": [np.quantile(gaps, 0.025)],
            "ci_high": [np.quantile(gaps, 0.975)],
            "bootstrap_draws": [draws],
        }
    )

def feature_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    numeric = frame.select_dtypes(include="number").drop(columns=[spec.score], errors="ignore")
    numeric = numeric.loc[:, numeric.nunique(dropna=True) > 1]
    if spec.score not in frame or numeric.empty:
        return pd.DataFrame(columns=["feature", "spearman_with_risk", "absolute_association"])
    corr = numeric.corrwith(frame[spec.score], method="spearman").dropna()
    return (
        corr.rename("spearman_with_risk")
        .rename_axis("feature")
        .reset_index()
        .assign(absolute_association=lambda d: d["spearman_with_risk"].abs())
        .sort_values("absolute_association", ascending=False)
        .head(12)
    )

def temporal_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    if not spec.date or spec.date not in frame:
        return pd.DataFrame(
            columns=["period", "records", "metric_mean", "risk_score", "rolling_risk"]
        )
    x = frame.dropna(subset=[spec.date]).copy()
    x["period"] = pd.to_datetime(x[spec.date]).dt.to_period("M").dt.to_timestamp()
    out = (
        x.groupby("period")
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
        .sort_values("period")
    )
    out["rolling_risk"] = out["risk_score"].rolling(3, min_periods=1).mean()
    out["risk_change"] = out["rolling_risk"].diff()
    return out

def sensitivity(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    rows = []
    for quantile in [0.70, 0.75, 0.80, 0.85, 0.90]:
        threshold = frame[spec.score].quantile(quantile)
        flagged = frame[frame[spec.score] >= threshold]
        rows.append(
            {
                "risk_quantile": quantile,
                "threshold": threshold,
                "flagged_records": len(flagged),
                "flagged_share": len(flagged) / len(frame),
                "flagged_metric_mean": flagged[spec.metric].mean(),
            }
        )
    return pd.DataFrame(rows)

def decision_queue(frame: pd.DataFrame, spec: AnalysisSpec, limit: int = 20) -> pd.DataFrame:
    columns = [spec.entity, spec.group, spec.metric, spec.score]
    if spec.date:
        columns.insert(1, spec.date)
    columns = list(dict.fromkeys(c for c in columns if c in frame))
    return frame.nlargest(limit, spec.score)[columns].reset_index(drop=True)

def shortage_concentration(frame: pd.DataFrame) -> pd.DataFrame:
    active = frame.loc[frame["active_flag"].eq(1)]
    by_drug = (
        active.groupby("generic_name", dropna=False)
        .agg(
            manufacturers=("company_name", "nunique"),
            active_records=("active_flag", "sum"),
            max_age_days=("shortage_age_days", "max"),
            escalation_score=("escalation_score", "max"),
        )
        .reset_index()
    )
    return by_drug.merge(
        concentration_sensitivity(frame), on="generic_name", validate="one_to_one"
    ).sort_values(["active_records", "max_age_days"], ascending=False)

def run_analysis(slug: str, frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    spec = SPECS[slug]
    outputs = {
        "data_audit": data_audit(frame, spec.entity),
        "segment_scorecard": segment_scorecard(frame, spec),
        "bootstrap_gap": bootstrap_top_gap(frame, spec),
        "feature_diagnostics": feature_diagnostics(frame, spec),
        "temporal_diagnostics": temporal_diagnostics(frame, spec),
        "threshold_sensitivity": sensitivity(frame, spec),
        "decision_queue": decision_queue(frame, spec),
    }
    if slug == "01_consumer_complaints":
        outputs["response_queue_intervals"] = monthly_queue(frame)
    elif slug == "03_sec_financial_health":

        outputs["deposit_outflow_scenario"] = bank_scenario(frame)
        outputs["threshold_sensitivity"] = threshold_sensitivity(frame)
    elif slug == "04_hospital_readmissions":
        outputs["observed_rate_uncertainty"] = observed_rate_review(frame)
        outputs["shrinkage_sensitivity"] = shrinkage_sensitivity(frame)
    elif slug == "05_drug_safety":
        outputs["concentration_analysis"] = shortage_concentration(frame)
        outputs["record_hhi_sensitivity"] = concentration_sensitivity(frame)
    elif slug == "07_electricity_affordability":
        outputs["growth_scenarios"] = energy_projection(frame)
        outputs["monthly_pressure_signals"] = monthly_signals(frame)
        outputs["persistence_sensitivity"] = persistence_sensitivity(frame)
    elif slug == "08_power_plant_emissions":
        outputs["abatement_pareto"] = emissions_pareto(frame)
        outputs["peer_scenarios"] = peer_scenarios(
            frame.rename(columns={"generation_mwh": "net_generation_mwh"}).drop(
                columns=["peer_count"], errors="ignore"
            )
        )
    elif slug == "10_semiconductor_supply_chain":
        outputs["divergence_analysis"] = supply_chain_divergence(frame)
        outputs.update(pulse_tables(frame))
    return outputs

analysis_outputs = run_analysis(SLUG, clean)
print("Calculated outputs:", {name: len(frame) for name, frame in analysis_outputs.items()})
display(analysis_outputs["concentration_analysis"].head(5))


pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


Calculated outputs: {'data_audit': 5, 'segment_scorecard': 59, 'bootstrap_gap': 1, 'feature_diagnostics': 5, 'temporal_diagnostics': 14, 'threshold_sensitivity': 5, 'decision_queue': 20, 'concentration_analysis': 70, 'record_hhi_sensitivity': 70}


,generic_name,manufacturers,active_records,max_age_days,escalation_score,active_package_records,known_manufacturers,unknown_company_records,known_record_hhi,record_hhi_lower,record_hhi_upper
38,Lisdexamfetamine Dimesylate Capsule,13,90,1170.0,76.9,90,13,0,0.077037,0.077037,0.077037
1,"Amphetamine Aspartate Monohydrate, Amphetamine...",11,73,1445.0,77.3,73,11,0,0.104147,0.104147,0.104147
36,Lidocaine Hydrochloride Injection,10,65,5330.0,77.5,65,10,0,0.200000,0.200000,0.200000
6,Bupivacaine Hydrochloride Injection,8,57,3140.0,78.1,57,8,0,0.204063,0.204063,0.204063
16,Dexmedetomidine Hydrochloride Injection,16,56,2360.0,76.6,56,16,0,0.105867,0.105867,0.105867


## 5 Check

Run every assertion and inspect the quality report. These checks apply to the current response, not to unsupplied historical downloads.

**Check before continuing:** every assertion passes and `passed` is true. Investigate missing keys, duplicates, impossible values, and material row-count changes before release.


In [5]:
KEY_COLUMNS = ["record_id"]
report = profile(clean)
assert report["passed"], report["notes"]
assert clean[KEY_COLUMNS].notna().all().all(), "Missing record key"
assert not clean.duplicated(KEY_COLUMNS).any(), "Duplicate record key"
assert len(raw) > 0 and len(clean) > 0
print("Current source rows:", len(raw), "clean rows:", len(clean))
print("Quality:", report)


Current source rows: 1600 clean rows: 1600
Quality: {'rows': 1600, 'columns': 22, 'duplicate_rows': 0, 'missing_cells': 3039, 'missing_rate': 0.086335, 'passed': True, 'notes': []}


## 6 Save

Run this cell only after the checks pass. It writes the dated extract, quality report, analytical tables, and source evidence under `data/manual_rebuild/`. If a fresh response has the same date as an existing rebuild, Save uses a source-digest suffix so the verified result is not overwritten; it refuses a conflicting partial folder.

**Check before continuing:** every named output exists in the dated folder; the source digest and reported row counts match the current response.


In [6]:
CURRENT_SOURCE_SHA = [download_report["sha256"]] if SLUG == "01_consumer_complaints" else [item["sha256"] for item in FETCH_LOG]
OUTPUT_DIR = PROJECT / "data" / "manual_rebuild" / SNAPSHOT_DATE
if OUTPUT_DIR.exists():
    prior_file = OUTPUT_DIR / "quality_report.json"
    prior_hashes = json.loads(prior_file.read_text(encoding="utf-8")).get("source_sha256") if prior_file.is_file() else None
    if isinstance(prior_hashes, str):
        prior_hashes = [prior_hashes]
    if prior_hashes != CURRENT_SOURCE_SHA:
        digest_suffix = hashlib.sha256(json.dumps(CURRENT_SOURCE_SHA).encode("utf-8")).hexdigest()[:12]
        OUTPUT_DIR = PROJECT / "data" / "manual_rebuild" / f"{SNAPSHOT_DATE}-{digest_suffix}"
        if OUTPUT_DIR.exists():
            raise FileExistsError(f"A different or partial rebuild already exists at {OUTPUT_DIR}; review it before saving")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean.to_csv(OUTPUT_DIR / "tableau_extract.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")
report.update({"project": SLUG, "source": "FDA drug shortages via openFDA",
               "sample_mode": False, "snapshot_date": SNAPSHOT_DATE,
               "source_sha256": [item["sha256"] for item in FETCH_LOG]})
(OUTPUT_DIR / "quality_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
analysis_dir = OUTPUT_DIR / "analysis"
analysis_dir.mkdir(exist_ok=True)
for name, frame in analysis_outputs.items():
    frame.to_csv(analysis_dir / f"{name}.csv", index=False, date_format="%Y-%m-%d")
(analysis_dir / "manifest.json").write_text(json.dumps({
    "project": SLUG, "sample_mode": False,
    "artifacts": {name: len(frame) for name, frame in analysis_outputs.items()}
}, indent=2), encoding="utf-8")
(OUTPUT_DIR / "source_manifest.json").write_text(json.dumps({
    "source_url": SOURCE_URL, "snapshot_date": SNAPSHOT_DATE,
    "responses": FETCH_LOG
}, indent=2), encoding="utf-8")
print("Saved manual rebuild to:", OUTPUT_DIR.relative_to(ROOT).as_posix())


Saved manual rebuild to: projects/drug_safety/data/manual_rebuild/2026-09-26


### Manual MySQL handoff after Save

If you are demonstrating the CDC-style database step, open this project's `sql/README.md` now. In MySQL Workbench, run `create_table.sql`, stage this run's checked `tableau_extract.csv`, run a temporary path-edited copy of `load_extract.sql`, then run `check_load.sql` and `review_queries.sql`. The SQL files create and check a mirror of the saved extract; they do not replace the notebook's calculations.

**Visible output:** Workbench shows the loaded row count, distinct source keys, missing-key checks, and a small inspection query. **Check before Show:** loaded rows equal this run's `quality_report.json` rows, distinct keys equal loaded rows, missing keys equal zero, and no unexplained load warnings remain. For a fresh response, use its own dated folder, not the pinned counts. If MySQL is unavailable, continue to Show with the notebook evidence and say explicitly that the SQL mirror was not tested.


## 7 Show

Display the rebuilt analytical table below. The packaged Tableau workbook in this project's tableau folder displays the pinned 2026-09-26 extract and is an additional saved view. Compare the saved table and dashboard before citing either; a fresh publisher response requires a deliberate workbook refresh and caption review.

**Check before continuing:** the displayed rows come from the current `analysis_outputs` and saved extract, with labels and units checked against this project's README.


In [7]:
display(analysis_outputs["concentration_analysis"].head(10))
print("Tableau input:", (OUTPUT_DIR / "tableau_extract.csv").relative_to(ROOT).as_posix())


,generic_name,manufacturers,active_records,max_age_days,escalation_score,active_package_records,known_manufacturers,unknown_company_records,known_record_hhi,record_hhi_lower,record_hhi_upper
38,Lisdexamfetamine Dimesylate Capsule,13,90,1170.0,76.9,90,13,0,0.077037,0.077037,0.077037
1,"Amphetamine Aspartate Monohydrate, Amphetamine...",11,73,1445.0,77.3,73,11,0,0.104147,0.104147,0.104147
36,Lidocaine Hydrochloride Injection,10,65,5330.0,77.5,65,10,0,0.200000,0.200000,0.200000
6,Bupivacaine Hydrochloride Injection,8,57,3140.0,78.1,57,8,0,0.204063,0.204063,0.204063
16,Dexmedetomidine Hydrochloride Injection,16,56,2360.0,76.6,56,16,0,0.105867,0.105867,0.105867
60,Ropivacaine Hydrochloride Injection,5,49,3109.0,80.0,49,5,0,0.285298,0.285298,0.285298
11,Clonazepam Tablet,6,40,1296.0,54.2,40,6,0,0.188750,0.188750,0.188750
18,Dextrose Monohydrate 5% Injection,4,36,1685.0,56.2,36,4,0,0.313272,0.313272,0.313272
44,"Methylphenidate Hydrochloride Tablet, Extended...",6,34,1158.0,79.2,34,6,0,0.202422,0.202422,0.202422
8,Carboplatin Injection,8,29,1247.0,78.1,29,8,0,0.162901,0.162901,0.162901


Tableau input: projects/drug_safety/data/manual_rebuild/2026-09-26/tableau_extract.csv


## 8 Decide

The next cell records a source-dated decision, saved evidence, and an interpretation limit for the pinned response. Review and revise both statements if you acquire a fresh response; do not carry this conclusion forward automatically.

**Final check:** the current quality check passed, each evidence file exists, and the conclusion is no stronger than the analysis supports.


In [8]:
DECISION_TEXT = "For the 2026-09-26 pinned FDA snapshot, use analysis/concentration_analysis.csv to prioritize manual verification of Current generic-name groups, beginning with lisdexamfetamine (90 listings) and amphetamine salts (73). The quality_report.json check passed; this is a registry screening queue, not a clinical severity ranking."
INTERPRETATION_LIMIT = "OpenFDA listing records are not patient exposure, supply volume, or harm. Generic-name and company labels require verification; record posting and update dates can lag the underlying shortage situation."
EVIDENCE_FILES = ["quality_report.json", "analysis/concentration_analysis.csv"]
assert report["passed"], "Prior quality check must pass"
assert DECISION_TEXT.strip(), "Write a decision supported by the rebuilt evidence"
assert INTERPRETATION_LIMIT.strip(), "State an interpretation limit"
missing = [name for name in EVIDENCE_FILES if not (OUTPUT_DIR / name).is_file()]
assert not missing, f"Missing evidence files: {missing}"
print("Source snapshot:", SNAPSHOT_DATE)
print("Evidence:", EVIDENCE_FILES)
print("Decision:", DECISION_TEXT)
print("Limit:", INTERPRETATION_LIMIT)


Source snapshot: 2026-09-26
Evidence: ['quality_report.json', 'analysis/concentration_analysis.csv']
Decision: For the 2026-09-26 pinned FDA snapshot, use analysis/concentration_analysis.csv to prioritize manual verification of Current generic-name groups, beginning with lisdexamfetamine (90 listings) and amphetamine salts (73). The quality_report.json check passed; this is a registry screening queue, not a clinical severity ranking.
Limit: OpenFDA listing records are not patient exposure, supply volume, or harm. Generic-name and company labels require verification; record posting and update dates can lag the underlying shortage situation.
